# Agent | Tools

**1. 도구 (Tool)**

https://docs.langchain.com/oss/python/integrations/tools/index#tools-and-toolkits

> "LLM이 사용할 수 있는 구체적인 기술이나 장비"

LLM은 기본적으로 학습된 데이터 내에서만 답변할 수 있으며, 실시간 정보나 정확한 수학 계산에는 취약하다. **Tool**은 이러한 한계를 보완하기 위해 LLM에게 쥐여주는 외부 기능이다.

* **역활:** 외부 API 호출, 웹 검색, 코드 실행, 파일 시스템 접근 등 LLM이 직접 할 수 없는 작업을 대신 수행한다.
* **예시:**
* `Google Search`: 최신 정보를 검색한다.
* `Calculator`: 정확한 수학 계산을 수행한다.
* `Python REPL`: 파이썬 코드를 작성하고 실행한다.



**2. 에이전트 (Agent)**

> "도구를 언제, 어떻게 사용할지 결정하는 두뇌"

**Agent**는 LLM을 추론 엔진(Reasoning Engine)으로 사용하여 사용자의 요청을 해결하기 위한 계획을 세우고 실행하는 주체이다. 단순히 정해진 코드를 순서대로 실행하는 것이 아니라, 상황에 따라 유연하게 행동을 결정한다.

* **역활:** 사용자의 질문을 분석하고, 어떤 **Tool**이 필요한지 판단(Thought)하고, 해당 도구를 실행(Action)한 뒤, 그 결과(Observation)를 보고 다음 행동을 결정하거나 최종 답변을 내놓는다.
* **작동 방식 (ReAct 패턴 예시):**
1. **질문:** "현재 서울 날씨에 맞는 옷차림 추천해줘."
2. **생각(Thought):** "서울의 현재 날씨를 먼저 알아야 한다." -> `Search` 도구 선택
3. **행동(Action):** `Search("서울 현재 날씨")` 실행
4. **관찰(Observation):** "서울 기온 5도, 맑음"이라는 결과 획득
5. **생각(Thought):** "5도면 코트나 패딩이 필요하다."
6. **최종 답변:** "현재 서울은 5도이므로 코트나 가벼운 패딩을 추천합니다."

**Agent와 Tools의 상호작용**

1. **Agent가 입력을 받음**: 사용자의 요청을 LLM으로 분석.
2. **적합한 Tool 선택**: 요청을 처리하는 데 가장 적합한 Tool을 선택.
3. **Tool 실행 및 결과 반환**: Tool을 실행하고 결과를 받아 사용자에게 응답.

In [1]:
# %pip install -Uqqq langchain_openai langchain_community langchain_tavily langgraph wikipedia numexpr 'arxiv<4' ddgs

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')


# Tools

In [3]:
import importlib, pkgutil # 모듈 동적 로드 / 패키지 탐색 유틸

package = importlib.import_module('langchain_community.tools')

# 해당 패키지 경로 아래의 하위 모듈들을 하나씩 순회
for module in pkgutil.iter_modules(package.__path__):
    print(module.name) # 각 모듈(도구) 이름 출력

ainetwork
amadeus
arxiv
asknews
audio
azure_ai_services
azure_cognitive_services
bearly
bing_search
brave_search
cassandra_database
clickup
cogniswitch
connery
convert_to_openai
databricks
dataforseo_api_search
dataherald
ddg_search
e2b_data_analysis
edenai
eleven_labs
few_shot
file_management
financial_datasets
github
gitlab
gmail
golden_query
google_books
google_cloud
google_finance
google_jobs
google_lens
google_scholar
google_serper
google_trends
graphql
human
ifttt
interaction
jina_search
jira
json
memorize
merriam_webster
metaphor_search
mojeek_search
multion
nasa
nuclia
office365
openai_dalle_image_generation
openapi
openweathermap
passio_nutrition_ai
playwright
plugin
polygon
powerbi
pubmed
render
requests
riza
scenexplain
searchapi
searx_search
semanticscholar
shell
slack
sleep
spark_sql
sql_database
stackexchange
steam
steamship_image_generation
tavily_search
vectorstore
wikidata
wikipedia
wolfram_alpha
yahoo_finance_news
you
youtube
zapier
zenguard


### Wikipedia Tool

In [5]:
from langchain_community.tools import WikipediaQueryRun       # 위키피디아 질문 실행 Tool
from langchain_community.utilities import WikipediaAPIWrapper # 위키피디아 검색/요약 API 래퍼 클래스

# 위키피디아 API래퍼를 Tool에 연결
wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

print(wiki_tool.run('physical AI')) # 위키피디아 검색/요약 결과

Page: Physical artificial intelligence
Summary: Physical artificial intelligence or physical AI refers to artificial intelligence (AI) systems that perceive, reason about and act within the physical world. These systems generally combine AI models with sensors, control systems, actuators and physical machines such as robots or autonomous vehicles. Physical AI overlaps with embodied artificial intelligence, robotics and autonomous systems, but it emphasizes the complete process of perceiving an environment, motion planning an action and physically executing the task to perform work. This differs from digital AI or generative AI (GenAI), which primarily stays in the information or digital realm.
The term became increasingly prominent during the AI boom in the 2020s as AI development expanded from primarily digital applications toward humanoid robots, self-driving vehicles, smart factories and other autonomous machines. Its boundaries are not standardized, and it is often treated as a con

In [6]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from pprint import pprint

messages = [('human', '걸그룹 튜이드의 멤버 알려줘')]

llm = init_chat_model('gpt-5.4-mini')
# print(llm.invoke('걸그룹 튜이드의 멤버 알려줘')) # 최신 정보 알지 못함

agent = create_agent(
    model = llm,
    tools = [wiki_tool]
)

response = agent.invoke({'messages': messages})
pprint(response)

{'messages': [HumanMessage(content='걸그룹 튜이드의 멤버 알려줘', additional_kwargs={}, response_metadata={}, id='5c9f9874-9d26-450e-b4de-18a5ee9eb5e4'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 174, 'total_tokens': 194, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHPSlRd4clCOgVv74j5ywdRlJtNXC', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04244-3b7c-7890-a2b6-4401f34e92b6-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': 'Tweed girl group members'}, 'id': 'call_23oZRmFoXHYobLwo

In [7]:
print(response['messages'][-1].content)

“튜이드”라는 걸그룹은 제가 바로 확인할 수 없었어요. 아마 **다른 이름을 잘못 적으신 것**일 수 있어요.

혹시 아래 중 하나를 말씀하신 걸까요?
- **트와이스(TWICE)**
- **뉴진스(NewJeans)**
- **르세라핌(LE SSERAFIM)**
- **ITZY**
- **IVE**
- **T-ara(티아라)**

원하시면 **정확한 그룹 이름**을 보내주시면 멤버를 바로 알려드릴게요.


### load_tools

**load_tools 사용가능 목록**

라이브러리를 통해 제공되는 외부 tool들을 langchain-community에서 통합하여 사용할 수 있다.
웬만한 기능들은 langchain 생태계 내에서 쉽게 쓸 수 있다.

https://docs.langchain.com/oss/python/integrations/providers/overview

https://docs.langchain.com/oss/python/integrations/tools

| 도구 이름        | 기능 예시               |
|------------------|------------------------|
| llm-math         | LLM 기반 수학 계산     |
| wikipedia        | 위키백과 검색          |
| serpapi          | 구글 검색 API          |
| requests_get     | HTTP GET 요청          |
| requests_post    | HTTP POST 요청         |
| arxiv            | arXiv 논문 검색        |
| pubmed           | PubMed 논문 검색       |
| dalle            | DALL-E 이미지 생성     |
| bing_search      | Bing 검색              |
| duckduckgo_search| DuckDuckGo 검색        |

### arxiv

In [8]:
from langchain_community.agent_toolkits.load_tools import load_tools

llm = init_chat_model('gpt-5.4-mini')
tools = load_tools(['arxiv','wikipedia'])
agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt = "당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해주세요."
)

messages = [('human', '(2608.26070) 이 논문의 내용을 간단하게 설명해줄래? (한글답변)')]
response = agent.invoke({'messages': messages})
pprint(response)
print("="*10)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='(2608.26070) 이 논문의 내용을 간단하게 설명해줄래? (한글답변)', additional_kwargs={}, response_metadata={}, id='955da975-ae82-47c0-b7f9-e3aa295f5d49'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 299, 'total_tokens': 320, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHPT0uomgyJqTMft4bQvs5AoxVJQT', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04244-76c2-71a0-b12f-bbadc7bcc710-0', tool_calls=[{'name': 'arxiv', 'args': {'query': '2608.26070'}, 'id': 'call_gPwuKaSP

In [9]:
import requests
import xml.etree.ElementTree as ET
from langchain_core.tools import tool

@tool
def search_arxiv(arxiv_id: str) -> str:
    """ arXiv 논문 ID로 제목, 저자, 초록을 조회합니다. """

    url = "https://export.arxiv.org/api/query"
    response = requests.get(
        url,
        params= {
            "id_list": arxiv_id,
            "max_results": 1
        },
        timeout = 10
    )

    response.raise_for_status()

    root = ET.fromstring(response.text)

    ns = {"atom": "http://www.w3.org/2005/Atom"}
    entry = root.find("atom:entry", ns)

    if entry is None:
        return "논문 정보를 찾을 수 없습니다."

    title = entry.findtext("atom:title", namespaces=ns).strip()
    summary = entry.findtext("atom:summary", namespaces=ns).strip()
    authors = [
        author.findtext("atom:name", namespaces=ns)
        for author in entry.findall("atom:author",ns)
    ]

    return f"""
제목: {title}
저자: {', '.join(authors)}
초록: {summary}
"""


In [10]:
tools = [search_arxiv, wiki_tool]

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt="당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해 주세요."
)

messages = [('human' '(2608.26070) 이 논문의 내용을 간단하게 설명해줄래? (한글답변)')]
response = agent.invoke({'messages': messages})
print(response['messages'][-1].content)

네. **2608.26070 논문 “Prefix Sliding for efficient test-time scaling”**를 아주 간단히 설명하면:

### 핵심 아이디어
이 논문은 **긴 추론을 할 때 메모리와 계산 비용을 줄이는 방법**을 제안합니다.  
보통 언어모델은 생각하는 동안 나온 **모든 중간 토큰을 계속 기억**하는데, 이게 길어질수록 매우 비싸집니다.

### 제안한 방법: Prefix Sliding
- **앞부분(prefix)**: 중요한 지시사항, 도구 사용법 같은 핵심 정보는 계속 유지
- **최근 토큰(window)**: 모델이 지금 막 생각한 내용도 유지
- **중간에 오래된 토큰들**: 중요도가 낮아지면 버려서 메모리 절약

즉, **“처음 중요한 것 + 최근 생각한 것만 남기고 나머지는 밀어내는 방식”**입니다.

### 왜 좋은가?
- 긴 추론을 해도 **메모리 사용량이 거의 늘지 않음**
- 별도 학습 없이도 기존 모델을 **최대 3배 빠르게** 만들 수 있음
- 강화학습과 함께 쓰면 **10만 토큰이 넘는 긴 추론**도 더 잘 처리할 수 있음
- 단순한 슬라이딩 윈도우나 중간 내용을 요약하는 방식보다 성능이 좋았다고 합니다

### 한 줄 요약
**긴 추론 중 오래된 중간 토큰을 버리고, 중요한 앞부분과 최신 토큰만 유지해서 더 효율적으로 오래 생각하게 만드는 방법**입니다.

원하시면 제가 이 논문을 **그림처럼 쉽게 비유해서**도 설명해드릴게요.


### llm-math

In [11]:
from langchain_community.agent_toolkits.load_tools import load_tools

llm = init_chat_model('gpt-4.1-mini')
# wikipedia, llm-math 도구 로드 (수학도구는 계산용 llm필요)
tools = load_tools(['wikipedia', 'llm-math'], llm=llm)

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt="당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답변해주세요. 단, 숫자계산은 llm-math 도구를 사용해서 답변에 활용해야 합니다."
)

response = agent.invoke({'messages': '3.5의 3제곱은 몇이야? 그리고 그 결과에 5를 곱해줘.'})
pprint(response)
print("="*10)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='3.5의 3제곱은 몇이야? 그리고 그 결과에 5를 곱해줘.', additional_kwargs={}, response_metadata={}, id='eec1d07f-e7dc-49a8-95f2-88baa5979e08'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 187, 'total_tokens': 207, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_ab8fa114f2', 'id': 'chatcmpl-EHPTBeFcfmu8TJbP0LNs2gqFhB9Ta', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04244-a296-7051-8974-8f2c27bafef8-0', tool_calls=[{'name': 'Calculator', 'args': {'__arg1': '3.5^3'}, 'id': 'call_4YXFs

### duckduckgo
https://docs.langchain.com/oss/python/integrations/tools/ddg

DuckDuckGo는 개인정보 추적 없이 웹 검색을 제공하는 검색 엔진으로,
LangChain에서는 이를 외부 최신 정보 검색용 Tool로 활용한다.

실시간 웹 검색 가능
→ LLM의 knowledge cutoff 이후 이슈(뉴스, 화제, 트렌드)에 대응 가능
로그인/API 키 불필요
→ 실습·교육 환경에서 바로 사용 가능
프라이버시 중심
→ 사용자 검색 이력 추적 없음

LangChain에서 제공하는 DuckDuckGo Tool 차이
- DuckDuckGoSearchRun
    - 검색 결과를 하나의 텍스트 요약으로 반환
    - 빠른 질의응답용에 적합
- DuckDuckGoSearchResults
    - 검색 결과를 리스트(제목, 링크, 스니펫 등 구조화) 형태로 반환
    - 에이전트가 여러 결과를 비교·판단해야 할 때 유리

In [12]:
# 덕덕고 검색 Tool 2종류
from langchain_community.tools import DuckDuckGoSearchRun, DuckDuckGoSearchResults

ddgs = DuckDuckGoSearchRun() # 검색 결과를 텍스트 요약 형태로 반환
print(ddgs.invoke("Trump's first name?"))  # 문자열 출력
ddgs2 = DuckDuckGoSearchResults() # 검색 결과를 제목/링크/스니펫 형태로 반환

print(ddgs2.invoke("Trump's first name?")) # 결과 출력



Donald Trump - Wikipedia For a chronological guide, see Timeline of the Donald Trump presidencies § First presidency (2017-2021). Trump was sworn in as president on January 20, 2017. During his first term, his administration focused on immigration, trade, tax cuts, and reducing government regulations. Trump withdrew the United States from the Trans-Pacific Partnership and announced that the country would leave the Paris Agreement on climate change. [13][14] He supported building a wall along the U.S.-Mexico border and ... Donald Trump is the 45th and 47th president of the United States (2017-21; 2025- ). Following his inauguration on January 20, 2025, Trump became only the second president to serve two nonconsecutive terms, the first being Grover Cleveland (1885-89; 1893-97). If you feel that a man's real last name is whatever last name his ancestors used first, then Trump's real last name might be Drumpf.
snippet: 6 hours ago - He launched side ventures, many licensing the Trump name,

In [13]:

llm = init_chat_model('gpt-5.4-mini')
# wikipedia, llm-math 도구 로드 (수학도구는 계산용 llm필요)
tools = [ddgs2]

agent = create_agent(llm, tools, system_prompt='모르는 정보가 있으면 ddgs tool을 사용해 검색해')

response = agent.invoke({'messages': 'gs25 민음사 빵'})
pprint(response)
print("="*10)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='gs25 민음사 빵', additional_kwargs={}, response_metadata={}, id='c2d64742-fc64-413f-ae70-2784b7f058cb'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 180, 'total_tokens': 207, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHPTP3enJpO4g1rGw1JaPVs4uDDFd', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04244-d90a-7120-9d80-a159b211d927-0', tool_calls=[{'name': 'duckduckgo_results_json', 'args': {'query': 'GS25 민음사 빵'}, 'id': 'call_aPtlydDDN4tuvvTYmgRby

### tavily-search

https://docs.langchain.com/oss/python/integrations/tools/tavily_search

In [ ]:
from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(
    max_result= 3,
    topic='general',
    include_images = True,
    search_depth = 'advanced'
)

tavily_tool.invoke('2026년 8월 현재 대한민국에서 가장 핫한 이슈가 뭐야?')


{'query': '2026년 8월 현재 대한민국에서 가장 핫한 이슈가 뭐야?',
 'follow_up_questions': None,
 'answer': None,
 'images': ['https://cdn.prod.website-files.com/66e955c4e382a96d6c5c0548/6a426f67d1b5fa839e454dfe_79_thumbnail2.png',
  'https://cdn.prod.website-files.com/66e955c4e382a96d6c5c0548/6a426e7fc594ca778cc36ab8_79_2-1.png',
  'https://cdn.prod.website-files.com/66e955c4e382a96d6c5c0548/6a426ed2d1b5fa839e44e068_79_2-2.png',
  'https://cdn.wakeupnews.co.kr/news/photo/202601/973_1856_5154.png',
  'https://i.ytimg.com/vi/spk_eNxCH_k/maxresdefault.jpg'],
 'results': [{'url': 'https://www.newsshin.co.kr/news/articleView.html?idxno=325554',
   'title': "뉴스신ㅣ2026년 8월 22일(토) ㅣ대한민국 '핫' 이슈",
   'content': '▣  AI·산업 최대 이슈  \n "AI가 명령을 기다리지 않는다면…인류는 준비됐나"  \n ▶ 브리핑  \n AI 모델이 인간의 직접 지시 없이 외부망 공격과 같은 행동을 할 수 있다는 우려가 제기되면서 미국 정치권에서 AI 통제 장치 논의가 급부상했다. 이른바 ‘AI 킬스위치 법’ 논의는 AI 안전 문제가 단순한 기술 문제가 아니라 국가 안보 문제로 이동했음을 보여준다.  \n → 구조 분석  \n AI 경쟁은 이제 "누가 더 강력한 AI를 만드는가"에서 "누가 AI를 통제할 수 있는가"의 싸움으로 변하고 있다.  \n 반도체·데이터센터·소프트

In [34]:
llm = init_chat_model('gpt-5.4-mini')

tools = [tavily_tool]

agent = create_agent(llm, tools, system_prompt='당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해 주세요.')

response = agent.invoke({'messages': '현재 AI업계에서 가장 핫한 주제가 뭐야?'})

pprint(response)
print("=" * 50)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='현재 AI업계에서 가장 핫한 주제가 뭐야?', additional_kwargs={}, response_metadata={}, id='980a084a-1c35-4dcb-bd15-2043c8dba3a8'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 1322, 'total_tokens': 1377, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHQhJQTaBUl9X0Yidc9XEanVF8WH7', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0428c-a230-7601-a351-736cdf39f7f9-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current hottest topics in AI industry 2026

In [ ]:

llm = init_chat_model('gpt-5.4-mini')
# wikipedia, llm-math 도구 로드 (수학도구는 계산용 llm필요)
tools = [ddgs2]

agent = create_agent(llm, tools, system_prompt="""
당신은 미국주식시장 분석봇입니다.
사용자가 요청한 기업에 대한 2026년 보고서를 직관적으로 분석해주세요.

# 출력형식
다음 내용을 포함해 표형식 출력 (분석기관별 레코드로 작성)

1. 분석기관명
2. 목표주가범위 (최저 ~ 최대)
3. 전망근거 키워드
4. 신뢰도 지수(1 ~ 10)
""")

response = agent.invoke({'messages': '2026년 애플 주가 전망 분석해 줘'})
pprint(response)
print("="*10)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='주가 분석해줘', additional_kwargs={}, response_metadata={}, id='0422b536-37de-4a71-b00d-03202ff2b8d4'),
              AIMessage(content='물론입니다. 다만 **어느 미국 기업**인지 알아야 2026년 기준으로 분석할 수 있습니다.\n\n아래 중 하나로 보내주세요:\n- **티커**: 예) AAPL, NVDA, TSLA\n- **회사명**: 예) 애플, 엔비디아, 테슬라\n\n원하시면 제가 바로 **표 형식**으로\n- 분석기관명\n- 목표주가범위\n- 전망근거 키워드\n- 신뢰도 지수\n\n까지 정리해드리겠습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 121, 'prompt_tokens': 260, 'total_tokens': 381, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHPTWYP6hJTDzG2i6fQP0D0EABlVN', 'service_tier': 'default', 'finis

In [16]:
from IPython.display import display, Markdown

display(Markdown(response['messages'][-1].content))

물론입니다. 다만 **어느 미국 기업**인지 알아야 2026년 기준으로 분석할 수 있습니다.

아래 중 하나로 보내주세요:
- **티커**: 예) AAPL, NVDA, TSLA
- **회사명**: 예) 애플, 엔비디아, 테슬라

원하시면 제가 바로 **표 형식**으로
- 분석기관명
- 목표주가범위
- 전망근거 키워드
- 신뢰도 지수

까지 정리해드리겠습니다.

### @tool

In [17]:
# eval / exec로 문자열 코드 실행
a = 10
print(eval("5 + 3 + a")) # 문자열을 평가해서 결과를 반환
exec("b = 10") # 문자열을 실행 (할당 가능)
print(b)

18
10


In [33]:
from langchain_core.tools import tool

@tool
def simple_calculator(query: str) -> str:
    """산술연산을 위한 간단한 계산기 Tool"""  # 함수 설명(1줄)
    """
    산술연산을 위한 간단한 계산기
    Args:
        query: 계산식
    Return:
        계산식 결과값

    Examples:
    - simple_calculator("5 + 3 - 2") -> "계산 결과: 6"
    - simple_calculator("4 ** 2 / 8") -> "계산 결과: 2"
    """
    try:
        result = eval(query)
        return f"계산 결과 : {result}"
    except Exception as e:
        return f"계산 오류 : {str:e}"
simple_calculator

StructuredTool(name='simple_calculator', description='산술연산을 위한 간단한 계산기 Tool', args_schema=<class 'langchain_core.utils.pydantic.simple_calculator'>, func=<function simple_calculator at 0x000002136BB0FA60>)

In [19]:

llm = init_chat_model('gpt-5.4-mini')
# wikipedia, llm-math 도구 로드 (수학도구는 계산용 llm필요)
tools = [simple_calculator]

agent = create_agent(llm, tools, system_prompt="당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답변해주세요. 단, 숫자계산은 llm-math 도구를 사용해서 답변에 활용해야 합니다.")

response = agent.invoke({'messages': '7 + 3 * 8 이거를 계산해줘'})
pprint(response)
print("="*10)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='7 + 3 * 8 이거를 계산해줘', additional_kwargs={}, response_metadata={}, id='1e72a76e-2113-4524-a5c1-d1cb7fd5c5b0'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 197, 'total_tokens': 221, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHPTXDUQNxiCCMkcmGmXWq7NhXAiQ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04244-f669-7ca3-8ce3-52dacc5f2029-0', tool_calls=[{'name': 'simple_calculator', 'args': {'query': '7 + 3 * 8'}, 'id': 'call_rjHqOQFxCYxOk1EsIoMZ

In [20]:
response = agent.invoke({'messages': '김치볶음밥 레시피?'})
pprint(response)
print("="*10)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='김치볶음밥 레시피?', additional_kwargs={}, response_metadata={}, id='f04b0b86-e1cd-45af-bdbe-95c027b1557f'),
              AIMessage(content='물론이죠! 기본적인 **김치볶음밥 레시피**를 간단하게 알려드릴게요.\n\n## 재료\n- 밥 1공기\n- 김치 1/2컵~1컵\n- 돼지고기나 햄, 참치 등 원하는 재료 조금\n- 대파 1/2대\n- 식용유 1~2큰술\n- 고춧가루 1작은술(선택)\n- 간장 1작은술\n- 설탕 1/2작은술(김치가 너무 시면)\n- 참기름 1작은술\n- 깨 약간\n- 계란 1개(선택)\n\n## 만드는 법\n1. **재료 손질**\n   - 김치는 잘게 썰고, 대파도 송송 썰어주세요.\n\n2. **볶기**\n   - 팬에 식용유를 두르고 대파를 먼저 볶아 향을 냅니다.\n   - 김치를 넣고 2~3분 정도 충분히 볶아주세요.\n   - 고기나 햄, 참치를 넣는다면 이때 같이 볶습니다.\n\n3. **밥 넣기**\n   - 밥을 넣고 잘 풀어가며 볶아주세요.\n   - 간장, 고춧가루, 설탕을 취향껏 넣어 간을 맞춥니다.\n\n4. **마무리**\n   - 불을 끄고 참기름, 깨를 넣어 섞어줍니다.\n   - 원하면 계란후라이를 올려서 완성!\n\n## 팁\n- **김치를 충분히 볶아야** 맛이 더 깊어져요.\n- 밥이 너무 질면 볶음밥이 질어질 수 있으니, **찬밥**을 쓰면 좋아요.\n- 김치가 많이 시다면 설탕을 아주 조금 넣으면 맛이 부드러워집니다.\n\n원하시면 제가 바로  \n**1인분 기준으로 더 정확한 계량**이나 **참치김치볶음밥 / 스팸김치볶음밥 버전**으로도 알려드릴게요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 484, 'promp

In [32]:
import json

OPENWEATHER_API_KEY = os.getenv('OPENWEATHER_API_KEY')

@tool
def get_current_weather(city="Seoul", units="metric"):
    """
    OpenWeather API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수

    Args:
        - city: str 날씨정보를 가져올 도시 이름. **반드시 영문으로 작성하세요.**
            - 변환예시:
                - 서울 -> Seoul
                - 충남, 충청남도 -> Chungcheongnam-do
                - 부산 -> Busan
        - units: str 온도단위를 설정하는 문자열
          - metric(기본값: 섭씨, 미터)
          - imperial(화씨, 야드)
    Return:
        - str: json 형식으로 변환된 현재 날씨 정보
    """

    url = f'https://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPENWEATHER_API_KEY}&units={units}'
    response = requests.get(url)
    data = response.json()  # json -> dict

    weather_info = {}

    if response.status_code == 200:  # 정상 응답 받은 경우
        weather_description = data['weather'][0]['description']  # 날씨 설명
        temp = data['main']['temp']  # 현재 기온
        temp_feels_like = data['main']['feels_like']  # 체감 온도
        humidity = data['main']['humidity']  # 습도

        weather_info = {
            'city': city,
            'description': weather_description,
            'temperature': temp,
            'temperature_feels_like': temp_feels_like,
            'humidity': humidity
        }

    else:  # 응답 불량
        weather_info = {
            'city': city,
            'description': 'Not Found',
            'temperature': 'Not Found',
            'temperature_feels_like': 'Not Found',
            'humidity': 'Not Found'
        }

    return json.dumps(weather_info)  # dict -> json

get_current_weather

StructuredTool(name='get_current_weather', description='OpenWeather API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수\n\nArgs:\n    - city: str 날씨정보를 가져올 도시 이름. **반드시 영문으로 작성하세요.**\n        - 변환예시:\n            - 서울 -> Seoul\n            - 충남, 충청남도 -> Chungcheongnam-do\n            - 부산 -> Busan\n    - units: str 온도단위를 설정하는 문자열\n      - metric(기본값: 섭씨, 미터)\n      - imperial(화씨, 야드)\nReturn:\n    - str: json 형식으로 변환된 현재 날씨 정보', args_schema=<class 'langchain_core.utils.pydantic.get_current_weather'>, func=<function get_current_weather at 0x000002136BB29A80>)

In [ ]:
llm = init_chat_model('gpt-5.4-mini')
tools = [simple_calculator,get_current_weather]

agent = create_agent(llm, tools, system_prompt="당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답변해주세요.")

response = agent.invoke({'messages': '오늘 뭐 입어야 돼? 나 서울 살아.'})
pprint(response)
print("="*10)
print(response['messages'][-1].content)


{'messages': [HumanMessage(content='오늘 뭐 입어야 돼? 나 서울 살아.', additional_kwargs={}, response_metadata={}, id='392a0f0d-2323-4f34-b497-c1927c1c73a4'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 364, 'total_tokens': 387, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHPTbpP1STDNupLIH8fF4xVAxjK5L', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04245-0684-75a3-b7e4-32bff046d868-0', tool_calls=[{'name': 'get_current_weather', 'args': {'city': 'Seoul', 'units': 'metric'}, 'id': 'call_JN

In [23]:
# 한국 기준 현재 날씨/시간을 반환하는 Tool
from datetime import datetime
from pytz import timezone

@tool
def get_current_datetime(format: str='%Y-%m-%d %H:%M:%S') -> str:
    """
    한국기준 현재시각정보를 반환하는 함수
    Args:
        format: 날짜/시각 형식 지정
    Return:
        현재시각 문자열

    get_current_datetime() -> "2026-01-15 12:18:32"
    """

    kst = timezone('Asia/Seoul')
    return datetime.now(kst).strftime(format) # 현재 서울 시간을 받아, format 형식의 문자열을 반환
get_current_datetime

StructuredTool(name='get_current_datetime', description='한국기준 현재시각정보를 반환하는 함수\nArgs:\n    format: 날짜/시각 형식 지정\nReturn:\n    현재시각 문자열\n\nget_current_datetime() -> "2026-01-15 12:18:32"', args_schema=<class 'langchain_core.utils.pydantic.get_current_datetime'>, func=<function get_current_datetime at 0x000002134C563F60>)

In [24]:

@tool
def calculate_age(today_date: str, birth_date: str) -> int:
    """
    오늘날짜, 생년월일을 입력받아 나이를 계산하는 도구
    Args:
        - today_date(str): 오늘 날짜 (yyyy-mm-dd형식)
        - birth_date(str): 생년월일 (yyyy-mm-dd형식)
    Return:
        - 계산된 만나이(int)
    """
    try:
        today = datetime.strptime (today_date, '%Y-%m-%d')
        birthday = datetime.strptime (birth_date, '%Y-%m-%d')

        age = today.year - birthday.year
        if (today.month, today.day) < (birthday.month, birthday.day):
            age -=1
        return age
    except ValueError:
        return "날짜 형식이 올바르지 않습니다. yyyy-mm-dd 형식으로 전달해주세요."

calculate_age.invoke({'today_date': '2026-08-27', 'birth_date':'1920-10-11'})

105

In [ ]:
llm = init_chat_model('gpt-5.4-mini')
tools = load_tools(['wikipedia']) + [get_current_datetime,calculate_age]

agent = create_agent(llm, tools, system_prompt="당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답변해주세요.")

response = agent.invoke({'messages': '트럼프 대통령의 현재 나이는?'}, config= {'recursion_limit':10})
pprint(response)
print("="*10)
print(response['messages'][-1].content)


{'messages': [HumanMessage(content='트럼프 대통령의 현재 나이는?', additional_kwargs={}, response_metadata={}, id='01e4cbb2-01ae-455b-a75e-3ffcb5b60ee2'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 366, 'total_tokens': 421, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHPTeP29H4A035WEJlB2P310wkZ2V', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04245-140a-7f90-a77d-3ed216f233e0-0', tool_calls=[{'name': 'get_current_datetime', 'args': {'format': '%Y-%m-%d'}, 'id': 'call_IahjQKATQJ5vjNbexIc

## Memory
agent의 checkpointer속성에 메모리객체를 대화내역을 저장한다.
- 임시저장 InMemorySaver()
- 영구저장 SqliteSaver()

### InMemorySaver

In [26]:
from langgraph.checkpoint.memory import InMemorySaver
llm = init_chat_model('gpt-5.4-mini')
tools = [TavilySearch()]
# 체크포인터를 전달하여 대화 상태 저장 가능하도록 에이전트 생성
agent = create_agent(llm, tools, checkpointer=InMemorySaver())

response = agent.invoke(
    input = {'messages': [('human', '안녕! 만나서 반갑다! 나는 min이라고 해. 넌 누구니?')]},
    config = {'configurable': {'thread_id': '100'}} # thread_id로 대화 식별
)

print(response['messages'][-1].content)

안녕 min! 만나서 반가워 😊  
나는 OpenAI가 만든 AI 어시스턴트야. 질문에 답하고, 글을 쓰고, 아이디어를 정리하고, 같이 생각해주는 역할을 해.

편하게 이야기해줘. 오늘은 내가 뭐 도와줄까?


### sqliteSaver

In [ ]:
# Langgraph 상태 저장을 SQLite로 영속화하여 저장하는 체크포인터 패키지
%pip install -Uqqq langgraph-checkpoint-sqlite

Note: you may need to restart the kernel to use updated packages.


In [28]:
from langgraph.checkpoint.sqlite import SqliteSaver
from pprint import pprint

llm = init_chat_model('gpt-5.4-mini')
tools = [TavilySearch()]

with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer.setup()

    agent = create_agent(llm, tools, checkpointer=checkpointer)

    response = agent.invoke(
        input = {'messages': [('human', 'Langchain에 대해 설명해줘.')]},
        config = {'configurable': {'thread_id': '100'}} # thread_id로 대화 식별
    )
    pprint(response)
    print("="*10)
    print(response['messages'][-1].content)
    print("="*10)
    response = agent.invoke(
        input = {'messages': [('human', 'Langgraph에 대해 설명해줘.')]},
        config = {'configurable': {'thread_id': '100'}} # thread_id로 대화 식별
    )
    pprint(response)
    print("="*10)
    print(response['messages'][-1].content)

{'messages': [HumanMessage(content='Langchain에 대해 설명해줘.', additional_kwargs={}, response_metadata={}, id='6289f995-0991-49b5-9f11-fbdad9061e96'),
              AIMessage(content='LangChain은 **대규모 언어 모델(LLM)을 더 쉽게 활용하기 위한 프레임워크**입니다.  \n한마디로 말하면, **“챗GPT 같은 모델을 그냥 호출하는 수준을 넘어, 실제 앱처럼 연결·조합·자동화할 수 있게 해주는 도구 모음”**이라고 볼 수 있어요.\n\n## 왜 필요한가?\nLLM을 직접 쓰면 보통 이런 일이 필요합니다:\n\n- 프롬프트를 잘 구성해야 함\n- 여러 번 대화한 내용을 기억해야 함\n- 외부 데이터베이스, 문서, API와 연결해야 함\n- 모델 출력 형식을 맞춰야 함\n- 여러 단계를 순서대로 실행해야 함\n\nLangChain은 이런 것들을 **표준화된 방식으로 연결**해 줍니다.\n\n---\n\n## LangChain이 해주는 일\n대표적으로 다음과 같은 기능이 있습니다.\n\n### 1. 프롬프트 관리\n프롬프트 템플릿을 만들어 재사용하기 쉽게 합니다.\n\n### 2. 체인(Chain) 구성\n여러 작업을 순서대로 연결할 수 있습니다.  \n예:\n- 질문 입력\n- 문서 검색\n- 관련 내용 요약\n- 최종 답변 생성\n\n### 3. 외부 데이터 연결\n문서, PDF, 웹페이지, DB, 검색엔진 등을 LLM과 연결할 수 있습니다.\n\n### 4. 메모리 관리\n대화형 앱에서 이전 대화 내용을 기억하게 할 수 있습니다.\n\n### 5. 에이전트(Agent) 기능\nLLM이 상황에 따라 도구를 선택해서 쓰게 할 수 있습니다.  \n예:\n- 계산기\n- 웹 검색\n- 데이터 조회\n- API 호출\n\n---\n\n## 핵심 개념\nLangChain을 이해할 때 자주 나오는 용어들이 있어요.\n\n### 

In [30]:
with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer.setup()

    agent = create_agent(llm, tools, checkpointer=checkpointer)

    response = agent.invoke(
        input = {'messages': [('human', '오케이 완전 이해했어! 그럼 니가 말해준 langchain, langgraph를 세 줄로 설명해줘')]},
        config = {'configurable': {'thread_id': '100'}} # thread_id로 대화 식별
    )
    pprint(response)
    print("="*10)
    print(response['messages'][-1].content)
    print("="*10)

{'messages': [HumanMessage(content='Langchain에 대해 설명해줘.', additional_kwargs={}, response_metadata={}, id='6289f995-0991-49b5-9f11-fbdad9061e96'),
              AIMessage(content='LangChain은 **대규모 언어 모델(LLM)을 더 쉽게 활용하기 위한 프레임워크**입니다.  \n한마디로 말하면, **“챗GPT 같은 모델을 그냥 호출하는 수준을 넘어, 실제 앱처럼 연결·조합·자동화할 수 있게 해주는 도구 모음”**이라고 볼 수 있어요.\n\n## 왜 필요한가?\nLLM을 직접 쓰면 보통 이런 일이 필요합니다:\n\n- 프롬프트를 잘 구성해야 함\n- 여러 번 대화한 내용을 기억해야 함\n- 외부 데이터베이스, 문서, API와 연결해야 함\n- 모델 출력 형식을 맞춰야 함\n- 여러 단계를 순서대로 실행해야 함\n\nLangChain은 이런 것들을 **표준화된 방식으로 연결**해 줍니다.\n\n---\n\n## LangChain이 해주는 일\n대표적으로 다음과 같은 기능이 있습니다.\n\n### 1. 프롬프트 관리\n프롬프트 템플릿을 만들어 재사용하기 쉽게 합니다.\n\n### 2. 체인(Chain) 구성\n여러 작업을 순서대로 연결할 수 있습니다.  \n예:\n- 질문 입력\n- 문서 검색\n- 관련 내용 요약\n- 최종 답변 생성\n\n### 3. 외부 데이터 연결\n문서, PDF, 웹페이지, DB, 검색엔진 등을 LLM과 연결할 수 있습니다.\n\n### 4. 메모리 관리\n대화형 앱에서 이전 대화 내용을 기억하게 할 수 있습니다.\n\n### 5. 에이전트(Agent) 기능\nLLM이 상황에 따라 도구를 선택해서 쓰게 할 수 있습니다.  \n예:\n- 계산기\n- 웹 검색\n- 데이터 조회\n- API 호출\n\n---\n\n## 핵심 개념\nLangChain을 이해할 때 자주 나오는 용어들이 있어요.\n\n### 

In [31]:
# SQLite 체크포인터(DB)의 특정 thread_id의 대화 메시지 조회
with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer_tuple = checkpointer.get_tuple({"configurable": {"thread_id": "100"}})

    checkpointer_data = checkpointer_tuple.checkpoint
    messages = checkpointer_data['channel_values']['messages']

    for i,message in enumerate(messages, 1):
        msg_type = getattr(message, 'type', message.__class__.__name__)
        print(f"{i}: [{msg_type}] {message.content}")
        print()
        

1: [human] Langchain에 대해 설명해줘.

2: [ai] LangChain은 **대규모 언어 모델(LLM)을 더 쉽게 활용하기 위한 프레임워크**입니다.  
한마디로 말하면, **“챗GPT 같은 모델을 그냥 호출하는 수준을 넘어, 실제 앱처럼 연결·조합·자동화할 수 있게 해주는 도구 모음”**이라고 볼 수 있어요.

## 왜 필요한가?
LLM을 직접 쓰면 보통 이런 일이 필요합니다:

- 프롬프트를 잘 구성해야 함
- 여러 번 대화한 내용을 기억해야 함
- 외부 데이터베이스, 문서, API와 연결해야 함
- 모델 출력 형식을 맞춰야 함
- 여러 단계를 순서대로 실행해야 함

LangChain은 이런 것들을 **표준화된 방식으로 연결**해 줍니다.

---

## LangChain이 해주는 일
대표적으로 다음과 같은 기능이 있습니다.

### 1. 프롬프트 관리
프롬프트 템플릿을 만들어 재사용하기 쉽게 합니다.

### 2. 체인(Chain) 구성
여러 작업을 순서대로 연결할 수 있습니다.  
예:
- 질문 입력
- 문서 검색
- 관련 내용 요약
- 최종 답변 생성

### 3. 외부 데이터 연결
문서, PDF, 웹페이지, DB, 검색엔진 등을 LLM과 연결할 수 있습니다.

### 4. 메모리 관리
대화형 앱에서 이전 대화 내용을 기억하게 할 수 있습니다.

### 5. 에이전트(Agent) 기능
LLM이 상황에 따라 도구를 선택해서 쓰게 할 수 있습니다.  
예:
- 계산기
- 웹 검색
- 데이터 조회
- API 호출

---

## 핵심 개념
LangChain을 이해할 때 자주 나오는 용어들이 있어요.

### Chain
작업들을 연결한 흐름입니다.

### Prompt Template
프롬프트를 변수화한 템플릿입니다.

### Retriever
관련 문서를 찾아오는 역할입니다.

### Vector Store
문서를 임베딩해서 저장하고 검색하는 저장소입니다.

### Agent
LLM이 어떤 도구를 쓸지 스스로 결정하는 구조입니다.

---

1️⃣ 세션(메모리) 유지 방식
예: store = {}, ChatMessageHistory, InMemorySaver
- 특징
    - 서버 메모리에만 대화 상태를 저장
    - 서버 재시작/재배포 시 모두 사라짐
    - 구현이 가장 단순하고 빠름
- 사용 시기
    - 실습 / 데모 / PoC
    - 단일 서버, 짧은 대화
    - “지금 이 세션에서만 기억하면 되는” 경우
- 장단점
    - ✅ 속도 빠름, 구현 쉬움
    - ❌ 서버 내려가면 기억 소멸
    - ❌ 멀티 서버(스케일아웃) 불가능

2️⃣ SQLite 체크포인터
예: SqliteSaver, checkpoint.db
- 특징
    - 로컬 파일(DB)에 대화 상태 저장
    - 서버 재시작해도 대화 복원 가능
    - 설정/운영 부담이 거의 없음
- 사용 시기
    - 1대 서버 운영
    - “재접속 시 대화 이어가기”가 중요한 서비스
    - 내부 도구, 사내용 챗봇, 파일 기반 서비스
- 장단점
    - ✅ 재시작해도 대화 유지
    - ✅ 설정 간단 (파일 하나)
    - ❌ 동시접속/대량 트래픽에 취약
    - ❌ 운영·분석·확장성 한계

3️⃣ RDB (MySQL / PostgreSQL 등)
실무에서 가장 많이 쓰는 방식

- 특징
    - 대화 내역을 정규화된 테이블로 저장
    - 여러 서버가 공유 DB 사용 가능
    - 사용자/세션/대화/이력 분석까지 가능
- 사용 시기
    - 실서비스(운영 환경)
    - 로그인 사용자 기반 챗봇
    - 고객지원, 상담, 금융, 헬스케어, 교육 서비스
- 장단점
    - ✅ 서버 여러 대에서도 동일한 대화 유지
    - ✅ 로그/분석/감사/리포트 가능
    - ✅ 권한·보안·백업 체계화 가능
    - ❌ 설계/운영 비용 존재

- 요약하자면  
메모리 세션    : 빠르고 간단  
SQLite 같은 파일형 DB : 재접속 기억 + 운영 부담 최소  
RDB    같은 관계형 데이터베이스 : 확장성, 안정성, 분석, 운영  

- 서비스 구상 단계에서  
사용자별 히스토리 관리  
문제 발생 시 감사 로그  
대화 품질/모델 성능 분석  
요약/임베딩/재검색(RAG) 연계  
개인화 서비스(추천, 성향 파악)  
이걸 하려면 RDB 또는 그 이상(이벤트 로그, 데이터 웨어하우스) 가 필요